<a href="https://colab.research.google.com/github/popojinny/UROP/blob/main/2025_%EC%9D%B4%ED%9B%84_%EC%A3%BC%EC%A0%9C_MCI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q bertopic openpyxl sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.0 MB/s eta 0:00:00


In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)

print("파일:", filename)
print("논문 수:", len(df))
print("열 이름:")
print(df.columns.tolist())

Saving 2025이후(포함).xlsx to 2025이후(포함).xlsx
파일: 2025이후(포함).xlsx
논문 수: 618
열 이름:
['num', 'Unnamed: 1', '년도', '제목', '영문제목', 'Unnamed: 5', '초록', '영문초록', '서지정보', 'DOI']


In [5]:
filtered_df = df.dropna(subset=['num', '영문초록'])
docs = (
    filtered_df["영문초록"]
    .astype(str)
    .tolist()
)

print("초록 수:", len(docs))

초록 수: 618


In [6]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    zeroshot_topic_list=[
        "Mild Cognitive Impairment (MCI)"
    ],
    zeroshot_min_similarity=0.70,
    min_topic_size=25,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(docs)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-08-16 13:35:19,952 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

2026-08-16 13:36:57,140 - BERTopic - Embedding - Completed ✓
2026-08-16 13:36:57,142 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-16 13:37:13,943 - BERTopic - Dimensionality - Completed ✓
2026-08-16 13:37:13,946 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2026-08-16 13:37:13,985 - BERTopic - Zeroshot Step 1 - Completed ✓
2026-08-16 13:37:13,991 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-16 13:37:14,044 - BERTopic - Cluster - Completed ✓
2026-08-16 13:37:14,053 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-16 13:37:14,328 - BERTopic - Representation - Completed ✓


In [7]:
df["MCI_topic"] = [
    "YES" if topic == 0 else "NO"
    for topic in topics
]

print(df["MCI_topic"].value_counts())

MCI_topic
NO     425
YES    193
Name: count, dtype: int64


In [8]:
total = len(df)

mci_count = (df["MCI_topic"] == "YES").sum()

mci_ratio = mci_count / total * 100

print(f"전체 논문 수: {total:,}")
print(f"MCI 주제 논문 수: {mci_count:,}")
print(f"MCI 주제 비율: {mci_ratio:.2f}%")

전체 논문 수: 618
MCI 주제 논문 수: 193
MCI 주제 비율: 31.23%


In [10]:
df["년도"] = pd.to_numeric(
    df["년도"],
    errors="coerce"
)

year_summary = (
    df.dropna(subset=["년도"])
    .groupby("년도")
    .agg(
        전체논문수=("MCI_topic", "size"),
        MCI논문수=("MCI_topic", lambda x: (x == "YES").sum())
    )
    .reset_index()
)

year_summary["MCI비율"] = (
    year_summary["MCI논문수"]
    / year_summary["전체논문수"]
    * 100
)

year_summary

,년도,전체논문수,MCI논문수,MCI비율
0,2025,394,122,30.964467
1,2026,224,71,31.696429
